In [2]:
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

In [3]:
# 1. Konfigurasi Path dan Eksperimen
folder_data = 'data_preprocessed'
folder_hasil = 'hasil_evaluasi'
os.makedirs(folder_hasil, exist_ok=True)

# List semua file fitur yang akan diuji
daftar_fitur = {
    "Raw_Image": "fashion_raw_data.npz",
    "PCA_32":    "fashion_pca_32dim.npz",
    "PCA_64":    "fashion_pca_64dim.npz",
    "PCA_128":   "fashion_pca_128dim.npz",
    "VAE_32":    "fashion_vae_32dim.npz",
    "VAE_64":    "fashion_vae_64dim.npz",
    "VAE_128":   "fashion_vae_128dim.npz",
}

# Model yang digunakan
models = {
    "SVM": SVC(kernel='rbf', random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=2000, random_state=42)
}

results_list = []

print("Memulai proses evaluasi...\n")

# 2. Looping Utama (Update bagian Load Data)
for nama_label, file_name in daftar_fitur.items():
    path_file = os.path.join(folder_data, file_name)
    
    if not os.path.exists(path_file):
        print(f"Skip: {file_name} tidak ditemukan.")
        continue
        
    # Load Data
    data = np.load(path_file)
    
    # SEKARANG SEMUA KEY SUDAH SERAGAM (X_train, X_test, y_train, y_test)
    X_train = data['X_train']
    X_test = data['X_test']
    y_train = data['y_train']
    y_test = data['y_test']
    
    print(f"Testing Fitur: {nama_label} ({X_train.shape[1]} Dimensi)")

    for model_name, clf in models.items():
        # Training
        clf.fit(X_train, y_train)
        
        # Prediksi
        y_pred = clf.predict(X_test)
        
        # Hitung Metrik (Metodologi 3.6)
        acc = accuracy_score(y_test, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)
        
        # Simpan hasil ke list
        results_list.append({
            "Fitur": nama_label,
            "Model": model_name,
            "Accuracy": acc,
            "F1-Score": f1,
            "Dimension": X_train.shape[1]
        })
        
        # --- Simpan Confusion Matrix ---
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title(f'CM: {nama_label} + {model_name}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        
        cm_filename = f"CM_{nama_label}_{model_name}.png"
        plt.savefig(os.path.join(folder_hasil, cm_filename))
        plt.close()

# 3. Tampilkan Tabel Ringkasan
df_results = pd.DataFrame(results_list)
print("\n" + "="*60)
print("HASIL AKHIR PERBANDINGAN")
print("="*60)
print(df_results.to_string(index=False))

# Simpan tabel ke CSV
df_results.to_csv(os.path.join(folder_hasil, 'summary_results.csv'), index=False)
print(f"\nSelesai! Semua Confusion Matrix dan tabel ringkasan ada di folder '{folder_hasil}'")

Memulai proses evaluasi...

Testing Fitur: Raw_Image (784 Dimensi)
Testing Fitur: PCA_32 (32 Dimensi)
Testing Fitur: PCA_64 (64 Dimensi)
Testing Fitur: PCA_128 (128 Dimensi)
Testing Fitur: VAE_32 (32 Dimensi)
Testing Fitur: VAE_64 (64 Dimensi)
Testing Fitur: VAE_128 (128 Dimensi)

HASIL AKHIR PERBANDINGAN
    Fitur              Model  Accuracy  F1-Score  Dimension
Raw_Image                SVM    0.8828  0.882265        784
Raw_Image       RandomForest    0.8764  0.874918        784
Raw_Image LogisticRegression    0.8438  0.843009        784
   PCA_32                SVM    0.8675  0.867032         32
   PCA_32       RandomForest    0.8583  0.857044         32
   PCA_32 LogisticRegression    0.8148  0.813726         32
   PCA_64                SVM    0.8771  0.876795         64
   PCA_64       RandomForest    0.8614  0.859854         64
   PCA_64 LogisticRegression    0.8335  0.832599         64
  PCA_128                SVM    0.8853  0.884901        128
  PCA_128       RandomForest    0